In [6]:
import pandas as pd
import numpy as np
from K_Nearest_Neighbors import K_Nearest_Neighbors_fix # Import class cuối cùng của chúng ta
from sklearn.model_selection import train_test_split, KFold, GridSearchCV # Import cả hai
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score # Vẫn dùng để so sánh cuối cùng
from sklearn.pipeline import Pipeline # <<< CÔNG CỤ QUAN TRỌNG NHẤT
import warnings
import time
# Tắt cảnh báo
warnings.filterwarnings("ignore", category=UserWarning)

In [7]:

print("--- BẮT ĐẦU QUY TRÌNH CHUẨN (Train/Test Split + K-Fold) ---")

# --- 0. Tải dữ liệu ---
column_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']
df = pd.read_csv("iris.data", header=None, names=column_names)
X = df.drop('class', axis=1).values
y = df['class'].values

# ==============================================================================
# BƯỚC 1: TÁCH "ĐỀ THI CUỐI KỲ" (DÙNG train_test_split)
# ==============================================================================
# Chia 150 mẫu thành 120 (ôn tập) và 30 (thi thật)
# (X_train_full, y_train_full) là bộ 120 câu để "ôn tập" và "tìm k"
# (X_test_final, y_test_final) là bộ 30 câu "thi thật" CẤT KỸ ĐI
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, 
    test_size=0.2,     # 20% cho "đề thi cuối kỳ"
    random_state=42,   # Đảm bảo kết quả cố định
    stratify=y         # Đảm bảo tỷ lệ các lớp trong 2 tập là như nhau
)

print(f"Đã chia dữ liệu: {len(y_train_full)} mẫu 'ôn tập', {len(y_test_final)} mẫu 'thi thật'.")


--- BẮT ĐẦU QUY TRÌNH CHUẨN (Train/Test Split + K-Fold) ---
Đã chia dữ liệu: 120 mẫu 'ôn tập', 30 mẫu 'thi thật'.


In [8]:

# ==============================================================================
# BƯỚC 2: TÌM 'k' TỐT NHẤT (DÙNG K-Fold trên 120 mẫu 'ôn tập')
# ==============================================================================
print("\n--- Bắt đầu giai đoạn 'Ôn tập' (Tìm k tốt nhất dùng K-Fold) ---")

k_values_to_try = [3, 5, 7, 9, 11] # Các giá trị k chúng ta muốn thử
k_fold_scores = {}             # Nơi lưu điểm trung bình của mỗi k
n_folds = 5                    # Dùng 5-fold cross-validation

for k in k_values_to_try:
    # Thiết lập giàn thử nghiệm K-Fold
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies = [] # Lưu điểm của 5 fold cho k này
    
    # Chạy 5 lần kiểm tra 15 phút
    # Chú ý: .split() được gọi trên X_train_full (120 mẫu)
    for train_idx, val_idx in kf.split(X_train_full):
        # 1. Lấy dữ liệu cho "bài kiểm tra 15 phút" này
        # (ví dụ: 96 câu học, 24 câu kiểm tra)
        X_train_fold, X_val_fold = X_train_full[train_idx], X_train_full[val_idx]
        y_train_fold, y_val_fold = y_train_full[train_idx], y_train_full[val_idx]

        # 2. Scaling (Rất quan trọng: CHỈ fit trên 96 câu học)
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler.transform(X_val_fold) # Chỉ transform

        # 3. Fit và Score
        knn = K_Nearest_Neighbors_fix(k=k)
        knn.fit(X_train_fold_scaled, y_train_fold)
        
        # Dùng hàm score() thủ công của class
        acc = knn.score(X_val_fold_scaled, y_val_fold) 
        fold_accuracies.append(acc)
    
    # 4. Tính điểm trung bình cho k này
    mean_acc = np.mean(fold_accuracies)
    k_fold_scores[k] = mean_acc
    print(f"  k = {k}: Điểm K-Fold trung bình = {mean_acc:.4f}")

# 5. Chọn ra k tốt nhất
best_k = max(k_fold_scores, key=k_fold_scores.get)
print(f"--- Giai đoạn 'Ôn tập' kết thúc: 'k' tốt nhất là {best_k} ---")



--- Bắt đầu giai đoạn 'Ôn tập' (Tìm k tốt nhất dùng K-Fold) ---
  k = 3: Điểm K-Fold trung bình = 0.9667
  k = 5: Điểm K-Fold trung bình = 0.9583
  k = 7: Điểm K-Fold trung bình = 0.9417
  k = 9: Điểm K-Fold trung bình = 0.9417
  k = 11: Điểm K-Fold trung bình = 0.9417
--- Giai đoạn 'Ôn tập' kết thúc: 'k' tốt nhất là 3 ---


In [9]:

# ==============================================================================
# BƯỚC 3: "THI THẬT" (DÙNG 'k' tốt nhất trên "Đề Thi Cuối Kỳ")
# ==============================================================================
print("\n--- Bắt đầu giai đoạn 'Thi Thật' (trên 30 mẫu cất riêng) ---")

# 1. Chuẩn bị mô hình cuối cùng
final_model = K_Nearest_Neighbors_fix(k=best_k)

# 2. Chuẩn bị scaler cuối cùng
# Lần này, chúng ta fit scaler trên TOÀN BỘ 120 câu 'ôn tập'
final_scaler = StandardScaler()
X_train_full_scaled = final_scaler.fit_transform(X_train_full)

# 3. Huấn luyện mô hình cuối cùng trên TOÀN BỘ 120 câu 'ôn tập'
final_model.fit(X_train_full_scaled, y_train_full)

# 4. Mang "Đề Thi Cuối Kỳ" (30 câu) ra chấm
# Chỉ .transform() bằng scaler đã fit ở trên
X_test_final_scaled = final_scaler.transform(X_test_final)

# 5. Tính điểm thi thật
final_accuracy = final_model.score(X_test_final_scaled, y_test_final)

print("\n=================================================")
print(f"ĐỘ CHÍNH XÁC CUỐI CÙNG (trên tập test 'thi thật'): {final_accuracy:.4f}")
print("=================================================")


--- Bắt đầu giai đoạn 'Thi Thật' (trên 30 mẫu cất riêng) ---

ĐỘ CHÍNH XÁC CUỐI CÙNG (trên tập test 'thi thật'): 0.9333


In [10]:

# ==============================================================================
# BƯỚC 1: TÁCH "ĐỀ THI CUỐI KỲ" (Vẫn y hệt)
# ==============================================================================
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Đã chia dữ liệu: {len(y_train_full)} mẫu 'ôn tập', {len(y_test_final)} mẫu 'thi thật'.")

# ==============================================================================
# BƯỚC 2: TẠO PIPELINE VÀ TINH CHỈNH TỰ ĐỘNG
# ==============================================================================

# 1. Tạo một 'Pipeline' (đường ống) đóng gói các bước
#    Tên các bước là 'scaler' và 'knn' (bạn có thể tự đặt)
pipe = Pipeline([
    ('scaler', StandardScaler()),              # Bước 1: Chuẩn hóa
    ('knn', K_Nearest_Neighbors_fix())   # Bước 2: Chạy mô hình KNN của bạn
])

# 2. Chỉ định các tham số để thử
#    Cú pháp: <tên_bước>__<tên_tham_số>
param_grid = {
    'knn__k': [3, 5, 7, 9, 11] # Yêu cầu GridSearchCV thử các giá trị k này
}

# 3. Khởi tạo GridSearchCV
#    - estimator: là CẢ ĐƯỜNG ỐNG (pipeline)
#    - param_grid: là các tham số
#    - cv=5: tự động chạy 5-fold K-Fold
#    - n_jobs=-1: dùng tất cả lõi CPU để chạy song song
tuner = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=5,
    n_jobs=-1 # Tăng tốc
)

# 4. CHẠY! 
#    Dòng .fit() này sẽ tự động làm MỌI THỨ:
#    - Chạy K-Fold 5 lần cho mỗi giá trị k
#    - Trong mỗi fold, tự động chạy scaler đúng cách (chống rò rỉ)
#    - Tìm ra k tốt nhất
#    - TỰ ĐỘNG huấn luyện lại Pipeline (với k tốt nhất) trên TOÀN BỘ 120 mẫu
print(f"\n--- Bắt đầu tìm tham số tốt nhất (GridSearchCV)...")
tuner.fit(X_train_full, y_train_full) # Đưa dữ liệu CHƯA SCALED vào

print(f"\nThông số tốt nhất tìm được: {tuner.best_params_}")
print(f"Điểm cross-val trung bình tốt nhất: {tuner.best_score_:.4f}")

# ==============================================================================
# BƯỚC 3: "THI THẬT"
# ==============================================================================
print("\n--- Bắt đầu giai đoạn 'Thi Thật' ---")

# Dùng mô hình tốt nhất (tuner) để chấm điểm trên "đề thi cuối kỳ"
# Tuner sẽ TỰ ĐỘNG scale X_test_final bằng scaler đã fit trên 120 mẫu
final_accuracy = tuner.score(X_test_final, y_test_final)

print("\n=================================================")
print(f"ĐỘ CHÍNH XÁC CUỐI CÙNG (trên tập test 'thi thật'): {final_accuracy:.4f}")
print("=================================================")

Đã chia dữ liệu: 120 mẫu 'ôn tập', 30 mẫu 'thi thật'.

--- Bắt đầu tìm tham số tốt nhất (GridSearchCV)...

Thông số tốt nhất tìm được: {'knn__k': 5}
Điểm cross-val trung bình tốt nhất: 0.9667

--- Bắt đầu giai đoạn 'Thi Thật' ---

ĐỘ CHÍNH XÁC CUỐI CÙNG (trên tập test 'thi thật'): 0.9333


c:\Users\Quan Phan\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [11]:
print("\n--- BẮT ĐẦU CÂU 2: NHẬN DẠNG KÝ TỰ (Letter Recognition) ---")

# --- 0. Tải dữ liệu ---
# URL của file dữ liệu từ trang UCI
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/letter-recognition/letter-recognition.data"

# Dữ liệu này có 1 cột class (ký tự) và 16 cột thuộc tính (features)
# Chúng ta cần đặt tên cột thủ công vì file gốc không có header
column_names = ['letter'] + [f'feature_{i}' for i in range(1, 17)]
df_letter = pd.read_csv(data_url, header=None, names=column_names)

print(f"Đã tải xong {len(df_letter)} mẫu.")
# print(df_letter.head()) # (Bạn có thể bỏ comment dòng này để xem 5 dòng đầu)

# Tách X (features) và y (target)
X_letter = df_letter.drop('letter', axis=1).values
y_letter = df_letter['letter'].values

# =============================================================================
# BƯỚC 1: TÁCH "ĐỀ THI CUỐI KỲ" (Train/Test Split)
# =============================================================================
# Dùng tỷ lệ 80/20 (giống Câu 1)
# 16.000 mẫu 'ôn tập', 4.000 mẫu 'thi thật'
X_train_full_let, X_test_final_let, y_train_full_let, y_test_final_let = train_test_split(
    X_letter, y_letter,
    test_size=0.2,
    random_state=42,
    stratify=y_letter # Rất quan trọng để giữ tỷ lệ 26 ký tự trong 2 tập
)

print(f"Đã chia dữ liệu: {len(y_train_full_let)} mẫu 'ôn tập', {len(y_test_final_let)} mẫu 'thi thật'.")


--- BẮT ĐẦU CÂU 2: NHẬN DẠNG KÝ TỰ (Letter Recognition) ---
Đã tải xong 20000 mẫu.
Đã chia dữ liệu: 16000 mẫu 'ôn tập', 4000 mẫu 'thi thật'.


In [12]:
# =============================================================================
# BƯỚC 2: TẠO PIPELINE VÀ TINH CHỈNH TỰ ĐỘNG (GridSearchCV)
# =============================================================================

print("\n--- Bắt đầu tìm tham số tốt nhất (GridSearchCV) cho Câu 2 ---")
print("CẢNH BÁO: Quá trình này sẽ mất vài phút do dữ liệu lớn (16.000 mẫu)...")

# 1. Tạo Pipeline (y hệt Câu 1)
# Chúng ta vẫn dùng class K_Nearest_Neighbors_fix đã import ở đầu file
pipe_letter = Pipeline([
    ('scaler', StandardScaler()),              # Bước 1: Chuẩn hóa
    ('knn', K_Nearest_Neighbors_fix())   # Bước 2: Chạy KNN
])

# 2. Chỉ định các tham số để thử
# Ta chỉ thử 3 giá trị k để tiết kiệm thời gian
param_grid_letter = {
    'knn__k': [3, 5, 7]
}

# 3. Khởi tạo GridSearchCV
# Ta dùng cv=3 (3-Fold) thay vì 5 để chạy nhanh hơn
tuner_letter = GridSearchCV(
    estimator=pipe_letter,
    param_grid=param_grid_letter,
    cv=3,       # Dùng 3-fold cross-validation
    n_jobs=-1,  # Dùng tất cả lõi CPU
    verbose=2   # In ra tiến trình để biết code đang chạy
)

# 4. CHẠY! (Dòng này sẽ mất vài phút)
start_time = time.time() # Bắt đầu đếm giờ
tuner_letter.fit(X_train_full_let, y_train_full_let) # Đưa 16.000 mẫu vào huấn luyện
end_time = time.time()

print(f"\n--- Đã huấn luyện xong! (Thời gian: {end_time - start_time:.2f} giây) ---")
print(f"Thông số tốt nhất tìm được: {tuner_letter.best_params_}")
print(f"Điểm cross-val trung bình tốt nhất (trên tập 'ôn tập'): {tuner_letter.best_score_:.4f}")


--- Bắt đầu tìm tham số tốt nhất (GridSearchCV) cho Câu 2 ---
CẢNH BÁO: Quá trình này sẽ mất vài phút do dữ liệu lớn (16.000 mẫu)...
Fitting 3 folds for each of 3 candidates, totalling 9 fits

--- Đã huấn luyện xong! (Thời gian: 38.05 giây) ---
Thông số tốt nhất tìm được: {'knn__k': 3}
Điểm cross-val trung bình tốt nhất (trên tập 'ôn tập'): 0.9376


In [13]:
# =============================================================================
# BƯỚC 3: "THI THẬT" (trên 4.000 mẫu cất riêng)
# =============================================================================
print("\n--- Bắt đầu giai đoạn 'Thi Thật' (Câu 2) ---")

# Dùng mô hình tốt nhất (tuner_letter) để chấm điểm trên 4.000 mẫu test
# tuner_letter sẽ tự động scale dữ liệu test trước khi dự đoán
final_accuracy_letter = tuner_letter.score(X_test_final_let, y_test_final_let)

print("\n=================================================")
print(f"ĐỘ CHÍNH XÁC CUỐI CÙNG (trên 4.000 mẫu test 'thi thật'): {final_accuracy_letter:.4f}")
print("=================================================")


--- Bắt đầu giai đoạn 'Thi Thật' (Câu 2) ---

ĐỘ CHÍNH XÁC CUỐI CÙNG (trên 4.000 mẫu test 'thi thật'): 0.9487


c:\Users\Quan Phan\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
